# Notebook 02 of 7 — Single-Name Deep Dive

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

In NB01 I met the plumbing. Now I'm going to put one name through it end-to-end. I picked {SYMBOL} — it's the largest single position in my basket, and 'is {SYMBOL} a good company' is exactly the kind of question I used to answer with a gut feel. Let's replace that.

By the end of this notebook we will be able to answer one question:

> *What do I actually think of {SYMBOL} — as a repeatable, phase-by-phase answer instead of a hot take?*


## 0. Why start with one name

There's a discipline I stole from a much better trader: **never analyze
a basket before you can analyze its members**. Portfolio-level
concentration numbers are meaningless if you don't have a per-name
opinion to concentrate on.

So this notebook builds the per-name opinion. The next one
(NB03) uses it.

*The code cell below runs the full 7-phase pipeline for {SYMBOL} once,
then we walk each phase individually below.*


### Fundamental vs technical — and why the pipeline uses both

Every guide splits stock analysis into two camps and then tells you which one is right. Both camps are useful; neither is sufficient. Here's the mental model I use across the 7 phases:

> **📖 Fundamental analysis** — evaluating a stock by the underlying business: earnings, growth, balance-sheet health, competitive position, cash flow, management. Phases 1, 2, 4, and 6 of the pipeline are fundamental. Answers "is this a good company at a fair price?" [Investopedia on fundamental analysis →](https://www.investopedia.com/terms/f/fundamentalanalysis.asp)

> **📖 Technical analysis** — evaluating a stock by the price + volume history alone: trend, momentum, support/resistance, chart patterns. Phase 3 is technical. Answers "what is the tape saying right now?" [Investopedia on technical analysis →](https://www.investopedia.com/terms/t/technicalanalysis.asp)

The pipeline uses both, in that order: fundamentals form the thesis, technicals form the *sanity check*. A great business trading at a fair valuation whose price has been in a six-month waterfall on rising volume is a signal that someone with better information than me has been selling. That's not a veto — it's an input to Phase 7's composite. The whole point of a composite score is that no single phase gets to unilaterally decide.

### Provider chain for this notebook (Track A / [#1430](https://github.com/prajoria/OpenBB/issues/1430))

NB02's pipeline runs exclusively on `fmp_cached` (my paid + cached tier).
That's the primary. But every fundamentals / price call in the platform
falls back through the same authoritative chain the series adopted in
[NB01 §2](./01-getting-started-and-providers.ipynb):

> **fmp_cached → fmp → cboe → sec (EDGAR) → yfinance (last-resort, personal-use)**

Concretely for this notebook:

| Data path used below | Primary | Free-authoritative fallback |
|---|---|---|
| Company profile + fundamentals | `fmp_cached` | SEC XBRL **company facts** (`sec`) |
| Prices / OHLCV | `fmp_cached` | `cboe` (EOD) |
| Analyst consensus / price target | `fmp_cached` | *no free authoritative source — gap; keep on `fmp`* |
| Everything else | `fmp_cached` | `sec` → `cboe` → yfinance (labeled) |

> **📖 SEC XBRL is as-filed.** Fundamentals from SEC EDGAR reflect what
> the company filed on the day it filed it. Restatements appear as
> *later* filings — the earlier value stays visible. That is a feature
> for point-in-time backtesting and a caveat for anyone reading a
> single row: "as of date X" ≠ "the current company view of date X".

No code cells below change under this PR — the pipeline is already
authoritative. The note above documents the chain so a reader knows
which fallback is next when a call raises.


In [ ]:
# [Phase B / NB02 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


In [ ]:
# [Phase B / NB02 §0] shared HTML rendering toolkit — one consistent look for
# EVERY section's *output* (not just its markdown). The goal: running the
# notebook top-to-bottom produces a fully readable, self-explaining report
# whose result panels cross-link to authoritative external references.
#
# Design rule for links: we link ONLY to STABLE external sources — Investopedia
# term pages, canonical texts — and NEVER to sibling notebooks or in-notebook
# cell anchors. Cell anchors and notebook filenames drift every time the code
# is reorganized; an Investopedia term URL is stable for years. So the output
# stays correct even after the notebook is refactored.
import html as _html
import math as _math
from IPython.display import HTML, display

# --- shared palette (matches the 7-phase card grid) ----------------------
_NB_ACCENT = "#7aa2f7"
_NB_GREEN = ("#16a34a", "rgba(34,197,94,.16)")
_NB_RED = ("#dc2626", "rgba(239,68,68,.16)")
_NB_AMBER = ("#d97706", "rgba(245,158,11,.18)")
_NB_GRAY = ("#6b7280", "rgba(127,127,127,.16)")
_NB_BLUE = (_NB_ACCENT, "rgba(122,162,247,.16)")
_NB_TONES = {"good": _NB_GREEN, "bad": _NB_RED, "warn": _NB_AMBER,
             "neutral": _NB_GRAY, "accent": _NB_BLUE}

# --- curated, STABLE external references (Investopedia term pages) --------
# Keyed by short slug; each is (label, url). Reused across every section so
# the same concept always links to the same authoritative page.
NB_LINKS = {
    "fundamental":  ("Fundamental analysis", "https://www.investopedia.com/terms/f/fundamentalanalysis.asp"),
    "technical":    ("Technical analysis", "https://www.investopedia.com/terms/t/technicalanalysis.asp"),
    "market_cap":   ("Market capitalization", "https://www.investopedia.com/terms/m/marketcapitalization.asp"),
    "shares_out":   ("Shares outstanding", "https://www.investopedia.com/terms/o/outstandingshares.asp"),
    "inst_own":     ("Institutional ownership", "https://www.investopedia.com/terms/i/institutionalownership.asp"),
    "peer_group":   ("Peer group", "https://www.investopedia.com/terms/p/peer-group.asp"),
    "owner_earn":   ("Owner's earnings", "https://www.investopedia.com/terms/o/ownersearnings.asp"),
    "roic":         ("ROIC", "https://www.investopedia.com/terms/r/returnoninvestmentcapital.asp"),
    "roe":          ("ROE", "https://www.investopedia.com/terms/r/returnonequity.asp"),
    "gross_profit": ("Gross profitability", "https://www.investopedia.com/terms/g/gross_profit_margin.asp"),
    "accruals":     ("Accruals / earnings quality", "https://www.investopedia.com/terms/a/accrualaccounting.asp"),
    "piotroski":    ("Piotroski F-score", "https://www.investopedia.com/terms/p/piotroski-score.asp"),
    "altman":       ("Altman Z-score", "https://www.investopedia.com/terms/a/altman.asp"),
    "atr":          ("Average True Range (ATR)", "https://www.investopedia.com/terms/a/atr.asp"),
    "stop_loss":    ("Stop-loss order", "https://www.investopedia.com/terms/s/stop-lossorder.asp"),
    "trailing_stop": ("Trailing stop", "https://www.investopedia.com/terms/t/trailingstop.asp"),
    "dcf":          ("Discounted cash flow (DCF)", "https://www.investopedia.com/terms/d/dcf.asp"),
    "margin_safety": ("Margin of safety", "https://www.investopedia.com/terms/m/marginofsafety.asp"),
    "ev_ebitda":    ("EV / EBITDA", "https://www.investopedia.com/terms/e/ev-ebitda.asp"),
    "price_target": ("Analyst price target", "https://www.investopedia.com/terms/p/pricetarget.asp"),
    "sharpe":       ("Sharpe ratio", "https://www.investopedia.com/terms/s/sharperatio.asp"),
    "max_dd":       ("Maximum drawdown", "https://www.investopedia.com/terms/m/maximum-drawdown-mdd.asp"),
    "beta":         ("Beta", "https://www.investopedia.com/terms/b/beta.asp"),
    "cvar":         ("Conditional VaR (CVaR)", "https://www.investopedia.com/terms/c/conditional_value_at_risk.asp"),
    "kelly":        ("Position sizing / Kelly", "https://www.investopedia.com/terms/p/positionsizing.asp"),
    "info_ratio":   ("Information ratio", "https://www.investopedia.com/terms/i/informationratio.asp"),
    "rel_strength": ("Relative strength", "https://www.investopedia.com/terms/r/relativestrength.asp"),
    "sector_rot":   ("Sector rotation", "https://www.investopedia.com/terms/s/sector-rotation.asp"),
    "composite":    ("Weighted (composite) score", "https://www.investopedia.com/terms/w/weightedaverage.asp"),
    "trade_plan":   ("Trading plan", "https://www.investopedia.com/terms/t/trading-plan.asp"),
    "staged_entry": ("Scaling into a position", "https://www.investopedia.com/terms/s/scaling.asp"),
    "regime":       ("Risk-on / risk-off regime", "https://www.investopedia.com/terms/r/risk-on-risk-off.asp"),
    "pickle":       ("Python object serialization (pickle)", "https://docs.python.org/3/library/pickle.html"),
    "regression_test": ("Regression testing", "https://en.wikipedia.org/wiki/Regression_testing"),
}


def _nb_esc(x):
    return _html.escape("" if x is None else str(x))


def _nb_fmt(v):
    """Human-format a scalar for a value cell. No '$' prefix (values may be
    counts, ratios, or dollars — we stay unit-neutral and just abbreviate
    large magnitudes with T/B/M)."""
    if v is None:
        return "<span style='opacity:.45'>n/a</span>"
    if hasattr(v, "item") and not isinstance(v, (list, tuple, dict, str)):
        try:
            v = v.item()
        except Exception:  # noqa: BLE001
            pass
    if isinstance(v, bool):
        return "yes" if v else "no"
    if isinstance(v, float):
        if _math.isnan(v):
            return "<span style='opacity:.45'>n/a</span>"
        a = abs(v)
        if a >= 1e12:
            return f"{v / 1e12:.2f}T"
        if a >= 1e9:
            return f"{v / 1e9:.2f}B"
        if a >= 1e6:
            return f"{v / 1e6:.2f}M"
        return f"{v:,.4g}"
    if isinstance(v, int):
        return f"{v:,}"
    s = str(v)
    return _nb_esc(s if len(s) <= 240 else s[:240] + "…")


def nb_pill(text, tone="neutral"):
    """A small coloured status badge."""
    fg, bg = _NB_TONES.get(tone, _NB_GRAY)
    return (
        f"<span style='display:inline-block;margin:2px 4px 2px 0;padding:2px 10px;"
        f"border-radius:10px;font:700 11px ui-sans-serif,system-ui;color:{fg};"
        f"background:{bg};white-space:nowrap'>{_nb_esc(text)}</span>"
    )


def _nb_links_footer(keys):
    chips = []
    for k in keys:
        pair = NB_LINKS.get(k)
        if not pair:
            continue
        label, url = pair
        chips.append(
            f"<a href='{url}' target='_blank' rel='noopener' "
            f"style='display:inline-block;margin:4px 6px 0 0;padding:3px 10px;"
            f"border-radius:12px;background:rgba(122,162,247,.12);color:{_NB_ACCENT};"
            f"font:12px/1.5 ui-sans-serif,system-ui;text-decoration:none'>"
            f"📖 {_nb_esc(label)}</a>"
        )
    if not chips:
        return ""
    return (
        "<div style='margin-top:10px;padding-top:8px;"
        "border-top:1px solid rgba(127,127,127,.18)'>"
        "<span style='font:600 11px ui-sans-serif,system-ui;opacity:.6;"
        "text-transform:uppercase;letter-spacing:.04em'>Learn more →</span><br>"
        + "".join(chips) + "</div>"
    )


def _nb_wrap(inner, tone=_NB_GRAY, max_height=460):
    fg = tone[0]
    return (
        f"<div style='max-height:{max_height}px;overflow:auto;padding:12px 14px;"
        f"margin:2px 0;border:1px solid rgba(127,127,127,.25);"
        f"border-left:4px solid {fg};border-radius:8px'>{inner}</div>"
    )


def nb_table(headers, rows):
    """Render a header + rows table. Cells may contain HTML (pills/links);
    callers are responsible for escaping their own text."""
    th = "".join(
        f"<th style='text-align:left;padding:4px 14px 5px 0;"
        f"font:600 11px ui-sans-serif,system-ui;opacity:.7;"
        f"text-transform:uppercase;letter-spacing:.03em;"
        f"border-bottom:1px solid rgba(127,127,127,.28)'>{_nb_esc(h)}</th>"
        for h in headers
    )
    body = ""
    for r in rows:
        tds = "".join(
            f"<td style='padding:3px 14px 3px 0;font:12px/1.5 ui-monospace,monospace;"
            f"border-bottom:1px solid rgba(127,127,127,.10)'>{c}</td>"
            for c in r
        )
        body += f"<tr>{tds}</tr>"
    return (
        "<table style='border-collapse:collapse;width:100%'>"
        f"<thead><tr>{th}</tr></thead><tbody>{body}</tbody></table>"
    )


def nb_panel(title, body_html, subtitle="", links=(), tone="neutral",
             badge=None, max_height=460):
    """Generic titled panel — the single look every section renders through."""
    fg, bg = _NB_TONES.get(tone, _NB_GRAY)
    pill = (
        f"<span style='margin-left:auto;padding:3px 12px;border-radius:12px;"
        f"font:700 11px ui-sans-serif,system-ui;letter-spacing:.03em;color:{fg};"
        f"background:{bg}'>{_nb_esc(badge)}</span>" if badge else ""
    )
    head = (
        "<div style='display:flex;align-items:center;gap:10px'>"
        f"<span style='font:700 15px ui-sans-serif,system-ui'>{_nb_esc(title)}</span>"
        f"{pill}</div>"
    )
    sub = (
        f"<div style='margin:3px 0 9px;font:13px/1.45 ui-sans-serif,system-ui;"
        f"opacity:.74'>{_nb_esc(subtitle)}</div>" if subtitle else ""
    )
    display(HTML(_nb_wrap(
        head + sub + body_html + _nb_links_footer(links),
        tone=(fg, bg), max_height=max_height,
    )))


def _nb_field_rows(obj, skip=("gate_passed", "gate_notes")):
    """Extract scalar-ish fields from a Phase result into (label, html) rows.
    DataFrames / dicts / long lists are summarized, not dumped."""
    rows = []
    for name in sorted(dir(obj)):
        if name.startswith("_") or name in skip:
            continue
        try:
            val = getattr(obj, name)
        except Exception:  # noqa: BLE001
            continue
        if callable(val):
            continue
        tname = type(val).__name__
        if tname == "DataFrame":
            disp = f"<span style='opacity:.6'>table · {val.shape[0]}×{val.shape[1]}</span>"
        elif isinstance(val, dict):
            disp = f"<span style='opacity:.6'>dict · {len(val)} keys</span>"
        elif isinstance(val, (list, tuple)) and len(val) > 6:
            disp = f"<span style='opacity:.6'>{tname} · {len(val)} items</span>"
        else:
            disp = _nb_fmt(val)
        rows.append((name, disp))
    return rows


def nb_render_phase(obj, num, name, blurb="", links=(), highlight=()):
    """Render one PhaseNResult as a consistent, cross-linked panel."""
    if obj is None:
        nb_panel(f"Phase {num} · {name}",
                 "<span style='opacity:.6'><i>no result (upstream failure)</i></span>",
                 tone="neutral", badge="—")
        return

    gate = getattr(obj, "gate_passed", None)
    tone = "good" if gate is True else "bad" if gate is False else "neutral"
    badge = "PASS" if gate is True else "FAIL" if gate is False else None

    chip_html = ""
    if highlight:
        chips = []
        for label, attr in highlight:
            chips.append(nb_pill(f"{label}: {_nb_fmt(getattr(obj, attr, None))}", "accent"))
        chip_html = f"<div style='margin:2px 0 8px'>{''.join(chips)}</div>"

    rows = _nb_field_rows(obj)
    table = nb_table(
        ["field", "value"],
        [(f"<span style='color:{_NB_ACCENT}'>{_nb_esc(k)}</span>", v) for k, v in rows],
    )

    notes = getattr(obj, "gate_notes", "") or ""
    note_html = ""
    if isinstance(notes, str) and notes.strip():
        trimmed = notes if len(notes) <= 600 else notes[:600] + "…"
        note_html = (
            "<div style='margin-top:9px;padding:7px 11px;border-radius:6px;"
            "background:rgba(127,127,127,.08);border-left:3px solid #7aa2f7;"
            "font:12px/1.5 ui-sans-serif,system-ui'>"
            f"<b>gate notes</b> — {_nb_esc(trimmed)}</div>"
        )

    body = f"<div style='margin-bottom:9px'>{_nb_esc(name)}</div>{chip_html}{table}{note_html}"
    nb_panel(f"Phase {num}", body, subtitle=blurb,
             links=links, tone=tone, badge=badge)


# tiny legend so the reader learns the colour language up front
display(HTML(_nb_wrap(
    "<b style='font:600 14px ui-sans-serif,system-ui'>Rendering toolkit ready</b>"
    "<div style='margin-top:7px;font:12px/1.7 ui-sans-serif,system-ui'>"
    "Every section below renders through one shared style. Colour language: "
    + nb_pill("PASS / Buy", "good") + nb_pill("FAIL / Avoid", "bad")
    + nb_pill("caution", "warn") + nb_pill("neutral", "neutral")
    + "<br>The <b>📖 Learn more</b> chips under each panel link to stable "
    "external references (Investopedia term pages, canonical docs) — never to "
    "sibling notebooks or cell anchors, which drift as the code is refactored."
    "</div>",
    tone=_NB_BLUE,
)))

In [ ]:
# [Phase B / NB02 §0] one shot — run all 7 phases on MSFT, walk them below
# Uses fmp_cached as the primary provider (see Analysis/stock_analysis.py
# PRIMARY_PROVIDER = "fmp_cached"). Typical wall-clock on a warm cache:
# 20-25 seconds. Cold cache: significantly slower (rerun to warm).
import sys, time, html, importlib
sys.path.insert(0, "../../Analysis")  # notebook CWD is notebooks/portfolio/
import stock_analysis
importlib.reload(stock_analysis)  # pick up readable __str__ without a kernel restart
from stock_analysis import AnalysisConfig, run_full_analysis
from IPython.display import HTML, display

t0 = time.perf_counter()
result = run_full_analysis(AnalysisConfig(symbol="MSFT"))
dt = time.perf_counter() - t0


# --- render the 7 phase summaries as color-coded cards -------------------
# Each PhaseNResult has a readable __str__ of the form
#   "Phase N · Name — metric · metric · ... · gate PASS"
# We parse that one-liner into (number, name, [metrics], status) and lay it
# out as a card: a number badge, the phase name, a color-coded status pill
# (green PASS / red FAIL·Avoid / amber Wait·Watch), metric chips, and any
# "override:" tail as a callout. Parsing the __str__ keeps this cell
# decoupled from each phase's field list; a fallback shows raw text if the
# shape ever changes.
def _status_color(text):
    t = (text or "").lower()
    if "pass" in t or "strong buy" in t or t.strip() == "buy":
        return ("#16a34a", "rgba(34,197,94,.16)")   # green
    if "fail" in t or "avoid" in t or "sell" in t or "reject" in t:
        return ("#dc2626", "rgba(239,68,68,.16)")    # red
    if any(w in t for w in ("hold", "watch", "wait", "n/a", "flat")):
        return ("#d97706", "rgba(245,158,11,.18)")   # amber
    return ("#6b7280", "rgba(127,127,127,.16)")      # gray


def _card(key, obj):
    if obj is None:
        return (
            "<div style='display:flex;gap:12px;align-items:center;padding:10px 12px;"
            "margin:6px 0;border:1px solid rgba(127,127,127,.25);border-left:4px solid "
            "#6b7280;border-radius:8px'>"
            "<span style='font:700 12px monospace;color:#6b7280'>{k}</span>"
            "<span style='opacity:.6'><i>no result (upstream failure)</i></span>"
            "</div>".format(k=html.escape(key))
        )
    parts = [s.strip() for s in str(obj).split(" · ")]
    number = parts[0].replace("Phase", "").strip() or key.upper()
    metrics = parts[1:]
    name = ""
    if metrics and " — " in metrics[0]:
        name, first = metrics[0].split(" — ", 1)
        metrics = ([first] + metrics[1:]) if first else metrics[1:]
    # pull a status pill: explicit "gate X", else the decision word (Phase 7)
    status, override = None, None
    if metrics and metrics[-1].lower().startswith("gate "):
        status = metrics[-1][5:].strip()
        metrics = metrics[:-1]
    elif number.strip() == "7" and metrics:
        status = metrics[0]           # action label (Avoid / Buy / ...)
        metrics = metrics[1:]
    override_bits = [m for m in metrics if m.lower().startswith("override:")]
    if override_bits:
        override = override_bits[0].split(":", 1)[1].strip()
        metrics = [m for m in metrics if not m.lower().startswith("override:")]
    fg, bg = _status_color(status or "")

    chips = "".join(
        "<span style='display:inline-block;padding:2px 8px;margin:2px 4px 2px 0;"
        "border-radius:10px;background:rgba(127,127,127,.12);font:12px/1.5 "
        "ui-monospace,monospace;white-space:nowrap'>{m}</span>".format(m=html.escape(m))
        for m in metrics
    )
    pill = (
        "<span style='margin-left:auto;padding:3px 12px;border-radius:12px;"
        "font:700 11px/1.4 ui-sans-serif,system-ui;letter-spacing:.03em;"
        "color:{fg};background:{bg};white-space:nowrap'>{s}</span>".format(
            fg=fg, bg=bg, s=html.escape(status.upper())) if status else "")
    note = (
        "<div style='margin-top:6px;padding:6px 10px;border-radius:6px;"
        "background:rgba(245,158,11,.10);border-left:3px solid #d97706;"
        "font:12px/1.45 ui-sans-serif,system-ui'>⚠ override — {o}</div>".format(
            o=html.escape(override)) if override else "")

    return (
        "<div style='padding:10px 12px;margin:6px 0;border:1px solid "
        "rgba(127,127,127,.25);border-left:4px solid {fg};border-radius:8px'>"
        "<div style='display:flex;align-items:center;gap:10px'>"
        "<span style='flex:0 0 auto;width:26px;height:26px;border-radius:50%;"
        "display:inline-flex;align-items:center;justify-content:center;"
        "background:rgba(122,162,247,.18);color:#7aa2f7;font:700 13px "
        "ui-sans-serif,system-ui'>{n}</span>"
        "<span style='font:600 14px/1.3 ui-sans-serif,system-ui'>{name}</span>"
        "{pill}</div>"
        "<div style='margin-top:6px'>{chips}</div>{note}</div>".format(
            fg=fg, n=html.escape(number), name=html.escape(name or key),
            pill=pill, chips=chips, note=note)
    )


cards = "".join(_card(k, result.get(k)) for k in sorted(result.keys()))
display(HTML(
    "<div style='max-height:520px;overflow:auto;padding:4px 2px'>"
    "<div style='display:flex;align-items:baseline;gap:10px;margin:2px 4px 8px'>"
    "<span style='font:700 15px ui-sans-serif,system-ui'>MSFT — 7-phase run</span>"
    "<span style='font:12px ui-monospace,monospace;opacity:.65'>"
    "{n} phases · {dt:.1f}s</span></div>{cards}</div>".format(
        n=len(result), dt=dt, cards=cards)
))


h:\masterswork\git\OpenBB-Portfolio-Validation\notebooks\portfolio\../../Analysis\stock_analysis.py:2026: RuntimeWarning: coroutine 'FMPCachedFinancialScoresFetcher.aextract_data' was never awaited
  logging.getLogger(__name__).debug(


## Reading the 7-phase map — the whole thesis before we zoom in

The colour-coded grid above *is* the answer, compressed to one screen. Before we open each phase's raw tables below, here is the mental model — written for a developer who is fluent in systems but still learning the market side.

Think of the pipeline as a **multi-stage validation chain with a final aggregator** — structurally identical to a CI/CD pipeline, with one deliberate difference:

- Each **phase** is an independent *check* that examines {SYMBOL} through exactly one lens and emits a **gate** (`PASS` / `FAIL`) plus a few headline metrics — the chips on each card.
- A `FAIL` is **not** a hard abort. Unlike CI, one red stage does **not** fail the build. Every gate result flows downstream into **Phase 7**, which computes a weighted **composite score** and collapses everything to one action word. This is the [ensemble principle](https://en.wikipedia.org/wiki/Ensemble_learning) from ML applied to investing: no single check gets a unilateral veto, because any single lens is wrong often enough that trusting it alone is a losing strategy.
- The one exception is Phase 7's **hard override** — a narrow safety rule (e.g. a bearish *weekly* trend) that *caps* a contribution rather than aborting. That is the amber ⚠ callout you saw on the Decision card.

> **📖 Gate** — in this pipeline a *gate* is a boolean quality bar for one phase: "did this lens clear its minimum standard?" It is a **signal, not a gatekeeper** — a red gate lowers the composite but does not stop the pipeline. Treat the seven gates like a pre-flight checklist where the pilot still gets to weigh a single failed item against everything else. [Investopedia: due diligence →](https://www.investopedia.com/terms/d/duediligence.asp)

> **📖 Composite score** — Phase 7's weighted blend of the six upstream lenses onto a single `0–5` scale, mapped to an action label (`Strong Buy` / `Buy` / `Hold·Watch` / `Avoid`). The weighting is deliberate: fundamentals set the thesis, technicals and peer-relative sanity-check it. See the [weighted-average](https://www.investopedia.com/terms/w/weightedaverage.asp) primer for the mechanics.

### The six lenses + the verdict, decoded

Read the card chips against this table. "Lens" = fundamental (the *business*) vs technical (the *tape*) vs quant (the *risk math*) — the split NB02's intro cell defined.

| # · Phase | Lens | The question it answers | Card chips, decoded | What makes the gate `PASS` | Learn the concept |
|---|---|---|---|---|---|
| **1 · Company & Tradeability** | Fundamental | *Who are they, and can I even trade this cleanly?* | `sector` · `cap` (market cap) · `peers` (comparable-company count) | A sector resolved **and** company-profile data came back non-empty — a pure *data-availability* gate. If Phase 1 fails, everything downstream is untrustworthy. | [Market cap](https://www.investopedia.com/terms/m/marketcapitalization.asp) · [Fundamental analysis](https://www.investopedia.com/terms/f/fundamentalanalysis.asp) |
| **2 · Fundamentals** | Fundamental | *Is the underlying business actually good over 5 years?* | `quality N/5` (weighted composite: 20% growth, 20% profitability, 20% capital-efficiency, 15% balance-sheet, 15% cash-flow, 10% structural) · `gross-profitability` (Novy-Marx) · `dilution` (5y share-count change) | `quality ≥ 3.5` **AND** [Sloan accruals](https://www.investopedia.com/terms/a/accrualaccounting.asp) `< 0.20` (an earnings-quality guard against paper profits). {SYMBOL} scored **3.47** — a 0.03 miss, so `FAIL`. | [Gross profitability](https://www.investopedia.com/terms/g/gross_profit_margin.asp) · [Share dilution](https://www.investopedia.com/terms/d/dilution.asp) |
| **3 · Technicals** | Technical | *What is the price/volume tape saying right now?* | `X/13 bullish` (setup conditions met) · `weekly` trend regime · `ATR` (volatility unit) · `entry` style | `≥ 6 of 13` conditions bullish **AND** outside the earnings blackout window (`> 5` trading days to earnings). A clean setup landing right before earnings still fails — event risk is a veto here. | [Technical analysis](https://www.investopedia.com/terms/t/technicalanalysis.asp) · [ATR](https://www.investopedia.com/articles/trading/08/atr.asp) |
| **4 · Valuation** | Fundamental | *Cheap, fair, or expensive — at today's price?* | `verdict` · `MOS` ([margin of safety](https://www.investopedia.com/terms/m/marginofsafety.asp) vs DCF fair value) · `Altman` (bankruptcy-risk Z) · `Piotroski N/9` (fundamental-momentum F-score) | `MOS` computed **AND** Altman Z `> 1.81` (not distress) **AND** verdict is *Undervalued* or *Fair Value* — i.e. **not Overvalued**. A negative MOS ⇒ price above fair value ⇒ `FAIL`. | [DCF](https://www.investopedia.com/terms/d/dcf.asp) · [Altman Z](https://www.investopedia.com/terms/a/altman.asp) · [Piotroski F](https://www.investopedia.com/terms/p/piotroski-score.asp) |
| **5 · Risk** | Quant | *If I own it, how much can it hurt — and how big a slice?* | `Sharpe` (return per unit risk) · `maxDD` (worst peak-to-trough) · `rec-size` (position size: lower of conviction vs [half-Kelly](https://www.investopedia.com/articles/trading/04/091504.asp)) · `fit` (Core / Satellite / Reject) | Portfolio fit is **not** `Reject`. Everything else (Sharpe, drawdown, VaR) feeds the *sizing*, not the pass/fail. | [Sharpe ratio](https://www.investopedia.com/terms/s/sharperatio.asp) · [Max drawdown](https://www.investopedia.com/terms/m/maximum-drawdown-mdd.asp) · [Position sizing](https://www.investopedia.com/terms/p/positionsizing.asp) |
| **6 · Peer-relative** | Fundamental + technical | *Good in isolation isn't enough — is it better than its peers?* | `score N/5` (5-block relative rank) · `IR` ([information ratio](https://www.investopedia.com/terms/i/informationratio.asp) vs the sector ETF) · `3m-rank` (percentile vs peers over 63 trading days) | Relative score `≥ 3.5`. This is where a great company in a *greater* sector cohort gets marked down — relative, not absolute. | [Information ratio](https://www.investopedia.com/terms/i/informationratio.asp) · [Relative strength](https://www.investopedia.com/terms/r/relativestrength.asp) |
| **7 · Decision** | Aggregator | *Given all six lenses, what do I actually do?* | `action` label · `composite N/5` · `entry` timing · optional ⚠ `override` | **No gate** — Phase 7 is the aggregator, not another check. It emits the verdict and the execution scaffold (stop, targets, staged entry, time-stop). | [Trade plan](https://www.investopedia.com/terms/t/trading-plan.asp) · [Stop-loss](https://www.investopedia.com/terms/s/stop-lossorder.asp) |

### The story the cards tell, in one breath

Phases **1–2** ask *is this a good company?* · **3** asks *is now a reasonable moment?* · **4** asks *at a fair price?* · **5** asks *at what size, given the pain it can inflict?* · **6** asks *versus the alternatives?* — and **7** fuses all of it into one number and one word. A single red gate is a data point, not a verdict; the verdict is Phase 7's job.

*Each section below opens one phase's raw tables so you can see exactly how every chip on its card was produced — starting with Phase 1.*


## 1. Phase 1 — Company

**Question:** *Who are they, what do they sell, how is the share structure?*

This is the boring phase and the most important one. If I can't explain in one paragraph what {SYMBOL} sells and to whom, I have no business owning it. Phase 1 forces me to.

*The code cell below renders phase 1's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*


> **📖 Shares outstanding** — total shares the company has issued and that are held by all shareholders (including insiders and restricted holders). Multiply by price for market cap. [Investopedia on shares outstanding →](https://www.investopedia.com/terms/o/outstandingshares.asp)

> **📖 Institutional ownership** — the fraction of shares held by mutual funds, pension funds, ETFs, and other institutions (reported quarterly via 13F filings, which NB04 covers in depth). Very high (>90%) means the price is largely set by big-money flows; very low (<20%) means retail sentiment can dominate. {SYMBOL} sits north of 70%. [Investopedia on institutional ownership →](https://www.investopedia.com/terms/i/institutionalownership.asp)

> **📖 Peer group** — the small set of companies you compare against because they operate in the same business, of roughly similar size, with roughly similar economics. Phase 1 picks the peer set; Phase 6 uses it to rank {SYMBOL}'s metrics. Bad peer set = bad relative-value verdict. [Investopedia on peer group →](https://www.investopedia.com/terms/p/peer-group.asp)


In [ ]:
# [Phase B / NB02 §1] Phase 1 — Company (rendered via the shared toolkit)
nb_render_phase(
    result["p1"], num=1, name="Company & Tradeability",
    blurb=("Who they are, the share structure, and whether the name is cleanly "
           "tradeable. A data-availability gate — if Phase 1 fails, nothing "
           "downstream is trustworthy."),
    highlight=[("sector", "sector"), ("market cap", "market_cap")],
    links=["fundamental", "market_cap", "shares_out", "inst_own", "peer_group"],
)

field,value
earnings_revision_3m_direction,down
free_float_pct,n/a
geo_df,table · 26×5
industry,
insider_df,table · 20×16
institutional_df,table · 1×32
market_cap,2.90T
metrics_df,table · 1×47
peers,list · 9 items
price_targets_df,table · 100×9


## 2. Phase 2 — Fundamentals

**Question:** *Are they earning real money, is it growing, is the balance sheet clean?*

The three-statement basics — plus the *ratios that catch nonsense*: owner earnings vs reported net income, cash-conversion, working-capital trend. If those look ugly, nothing later in this pipeline saves the pick.

*The code cell below renders phase 2's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

### Fundamentals — the four numbers that survive most audits

Phase 2 pulls dozens of ratios and hands them to Phase 7 as a health-score. Four of them do most of the load-bearing work:

> **📖 Owner earnings** — Warren Buffett's alternative to reported net income: net income + depreciation & amortization − maintenance capex − working-capital changes. Aims to be "the cash a shareholder could actually take out of the business without impairing it." Reported EPS can be gamed by accounting choices; owner earnings is a lot harder to inflate. [Investopedia on owner's earnings →](https://www.investopedia.com/terms/o/ownersearnings.asp)

> **📖 ROIC (return on invested capital)** — after-tax operating profit divided by the capital (equity + interest-bearing debt) tied up in the business. Answers: for every dollar the business has consumed, how many cents does it produce annually? Above the company's cost of capital = creating value; below = destroying value. [Investopedia on ROIC →](https://www.investopedia.com/terms/r/returnoninvestmentcapital.asp)

> **📖 ROE (return on equity)** — net income divided by shareholder equity. Simpler and older than ROIC; noisier because it can be inflated by leverage. Read ROE and ROIC together — a widening gap usually means the company is borrowing to buy back stock. [Investopedia on ROE →](https://www.investopedia.com/terms/r/returnonequity.asp)

> **📖 Piotroski F-score** — a 9-point checklist of accounting-quality signals (profitability, leverage, operating efficiency). 8 or 9 = clean books, low probability of a shock. 0-2 = something is off. Cheap and blunt; catches distress that individual ratios miss. [Investopedia on the Piotroski F-score →](https://www.investopedia.com/terms/p/piotroski-score.asp)

> **📖 Altman Z-score** — a distress-prediction score, originally calibrated on manufacturers. Above 3 = safe zone; below 1.8 = distress zone; the middle is grey. Not perfect on modern asset-light businesses (software, services) but still a useful "is bankruptcy remotely on the table?" gate. [Investopedia on the Altman Z-score →](https://www.investopedia.com/terms/a/altman.asp)

In [ ]:
# [Phase B / NB02 §2] Phase 2 — Fundamentals (rendered via the shared toolkit)
nb_render_phase(
    result["p2"], num=2, name="Fundamentals — earnings quality & balance-sheet health",
    blurb=("Five years of statements distilled to a 0–5 quality score. "
           "Gate: score ≥ 3.5 AND Sloan accruals < 0.20 (an earnings-quality "
           "guard against paper profits)."),
    highlight=[("quality", "score"), ("gross-profitability", "gross_profitability"),
               ("5y dilution", "dilution_5y")],
    links=["fundamental", "owner_earn", "roic", "roe", "gross_profit",
           "accruals", "piotroski", "altman"],
)

field,value
accruals_ratio,n/a
balance_df,table · 5×61
cash_df,table · 5×47
dilution_5y,0.01167
gross_profitability,0.3717
income_df,table · 5×39
kpi_df,table · 1×20
operating_leverage,n/a
ratios_df,table · 1×64
roe_decomp_df,table · 0×0


## 3. Phase 3 — Technicals

**Question:** *What is the tape saying — trend, momentum, volume?*

I don't trade on technicals. I *sanity-check* on technicals. If fundamentals say 'great, buy' and price has been in a 6-month waterfall on rising volume, someone knows something I don't.

*The code cell below renders phase 3's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

> **📖 ATR (average true range)** — a rolling 14-day average of the daily true range (max of today's high-low, |today's high − yesterday's close|, |today's low − yesterday's close|). Reads out as a dollar amount of typical daily volatility. Phase 7's staged-entry protocol sizes the trailing stop in units of ATR so the stop scales with the name's actual volatility instead of an arbitrary percentage. [Investopedia on ATR →](https://www.investopedia.com/terms/a/atr.asp)

> **📖 Stop-loss order** — a resting order that sells the position if price falls to a level you pre-committed to. Turns "how much am I willing to lose on this?" from a heat-of-the-moment decision into a policy. [Investopedia on stop-loss →](https://www.investopedia.com/terms/s/stop-lossorder.asp)

> **📖 Trailing stop** — a stop-loss that ratchets up (never down) as price rises. Locks in gains without capping upside. Phase 7 emits both the initial stop and the trailing rule in one bundle for NB05 to translate into broker orders. [Investopedia on trailing stops →](https://www.investopedia.com/terms/t/trailingstop.asp)

In [ ]:
# [Phase B / NB02 §3] Phase 3 — Technicals (rendered via the shared toolkit)
nb_render_phase(
    result["p3"], num=3, name="Technicals — trend, momentum, volume",
    blurb=("The tape as a sanity check, not a trigger. Gate: ≥ 6 setup "
           "conditions bullish AND outside the earnings blackout window "
           "(> 5 trading days to the next report)."),
    highlight=[("bullish", "bullish_count"), ("entry", "entry_quality"),
               ("ATR", "atr")],
    links=["technical", "atr", "stop_loss", "trailing_stop"],
)

field,value
atr,11.83
bullish_count,6
days_to_earnings,0
earnings_safe_window,no
entry_quality,Standard
extended_panel,n/a
fib_levels,dict · 5 keys
price_df,table · 312×37
signals,dict · 13 keys
weekly_trend_bullish,no


## 4. Phase 4 — Valuation

**Question:** *Is the price sensible against intrinsic and relative anchors?*

Multi-anchor: a DCF I don't trust in isolation, a relative-multiple band vs. peer set, and an owner-earnings yield vs. the risk-free rate. If two of three agree the name is overpriced, that's the input to phase 7, not a veto by itself.

*The code cell below renders phase 4's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

### Multi-anchor valuation — why no single number wins

Phase 4 refuses to output a single "fair value." It outputs a *band* built from three independent anchors, because every single valuation method has a known failure mode and the failures don't overlap. Sam's rule: if two of three anchors agree the name is expensive, that's a real vote toward expensive.

> **📖 DCF (discounted cash flow)** — project the business's future free cash flows out ~10 years, discount each year back to today using a required-return rate, sum. Result = "what this business is worth if my forecast + discount rate are right." Extremely sensitive to the terminal-growth and discount-rate inputs — small tweaks swing the answer wildly. Never a standalone verdict. [Investopedia on DCF →](https://www.investopedia.com/terms/d/dcf.asp)

> **📖 Enterprise value (EV)** — market cap + total debt − cash & equivalents. What you'd have to pay to buy the whole business free of its capital structure. Used as the numerator in cross-company valuation multiples because it's structure-neutral. [Investopedia on enterprise value →](https://www.investopedia.com/terms/e/enterprisevalue.asp)

> **📖 EV / EBITDA** — enterprise value divided by earnings before interest, taxes, depreciation, and amortization. The go-to "how expensive is this business's operating engine?" multiple. Comparable across capital structures in a way P/E is not. [Investopedia on EV/EBITDA →](https://www.investopedia.com/terms/e/ev-ebitda.asp)

> **📖 EV / Sales** — enterprise value divided by revenue. Blunter than EV/EBITDA; the fallback when a business is unprofitable or when EBITDA is being aggressively adjusted. Used to sanity-check high-growth names. [Investopedia on EV/Sales →](https://www.investopedia.com/terms/e/enterprisevaluesales.asp)

> **📖 Price target + analyst rating** — sell-side analysts publish a 12-month price target and a Buy/Hold/Sell rating; the *consensus* is the average across analysts covering the name. Useful as a sanity check on where the crowd is, not as a directional signal — the consensus is a lagging aggregate and the dispersion (max − min) often carries more information than the mean. [Investopedia on price targets →](https://www.investopedia.com/terms/p/pricetarget.asp) · [Investopedia on analyst ratings →](https://www.investopedia.com/terms/a/analystratings.asp)

In [ ]:
# [Phase B / NB02 §4] Phase 4 — Valuation (rendered via the shared toolkit)
nb_render_phase(
    result["p4"], num=4, name="Valuation — multi-anchor fair value",
    blurb=("DCF plus relative multiples, never a single number. Gate: margin "
           "of safety computed, Altman Z > 1.81 (not distress), and the "
           "verdict is not Overvalued."),
    highlight=[("verdict", "valuation_verdict"), ("MOS", "margin_of_safety"),
               ("Altman", "altman"), ("Piotroski", "piotroski")],
    links=["dcf", "margin_safety", "ev_ebitda", "price_target", "altman",
           "piotroski"],
)

field,value
altman,n/a
dcf_fair_value,151.7
entry_recommendation,Avoid — overvalued regardless of technicals
historical_multiples_df,table · 1×4
implied_growth,0.2924
margin_of_safety,-1.593
multiples_df,table · 1×10
multiples_vs_median,dict · 4 keys
peg_ratio,n/a
piotroski,n/a


## 5. Phase 5 — Risk

**Question:** *What happens if I am wrong — drawdown, vol, correlation?*

Not risk in isolation — risk *relative to what I already own*. {SYMBOL}'s beta only matters once I know NB03's basket-level exposure. This phase produces the numbers NB03 will merge in.

*The code cell below renders phase 5's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*


> **📖 Beta** — the slope of the regression of a stock's returns against a market index's returns. Beta = 1 means the stock moves 1-for-1 with the market on average; beta = 1.4 means 40% more amplitude; negative beta means it tends to move opposite. Not a promise about tomorrow — a description of the past correlation window. [Investopedia on beta →](https://www.investopedia.com/terms/b/beta.asp)

> **📖 Conditional Value at Risk (CVaR / expected shortfall)** — the average loss on the *worst* p% of historical days (e.g. CVaR-5% = mean loss on the worst 5% of days). Value-at-Risk gives you the threshold; CVaR gives you the average of the tail beyond the threshold, which is what actually happens when things go wrong. [Investopedia on CVaR →](https://www.investopedia.com/terms/c/conditional_value_at_risk.asp)

In [ ]:
# [Phase B / NB02 §5] Phase 5 — Risk (rendered via the shared toolkit)
nb_render_phase(
    result["p5"], num=5, name="Risk — drawdown, tail risk, position sizing",
    blurb=("What happens if I'm wrong, and the size that survives it. Gate: "
           "portfolio fit is not Reject. Sharpe / drawdown / CVaR feed the "
           "sizing, not the pass/fail."),
    highlight=[("Sharpe", "sharpe"), ("max drawdown", "max_drawdown"),
               ("rec size", "recommended_size"), ("fit", "portfolio_fit")],
    links=["sharpe", "max_dd", "beta", "cvar", "kelly"],
)

field,value
beta,0.8041
beta_down,0.7522
beta_up,0.5885
calmar,0.09004
conviction_size,0.01
cvar_95,-0.03773
gain_to_pain,1.022
half_kelly_size,0.0002203
kelly_fraction,0.01102
kurtosis,5.424


## 6. Phase 6 — Peer-relative

**Question:** *How does {SYMBOL} rank against its peer set on every metric that mattered?*

Same metrics as phases 2, 4, 5 — but ranked against the peer set from phase 1. This is where '{SYMBOL} looks cheap' either survives or dies.

*The code cell below renders phase 6's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*


In [ ]:
# [Phase B / NB02 §6] Phase 6 — Peer-relative (rendered via the shared toolkit)
nb_render_phase(
    result["p6"], num=6, name="Peer-relative -- ranked against its own peer set",
    blurb=("Good in isolation isn't enough. The name is scored against the "
           "Phase 1 peer group. Gate: relative score >= 3.5 -- it must be a "
           "leader, not just a survivor."),
    highlight=[("relative", "relative_score"), ("3m peer-rank", "rolling_3m_rank"),
               ("info ratio", "information_ratio")],
    links=["info_ratio", "rel_strength", "peer_group", "sector_rot"],
)

field,value
corr_matrix,table · 11×11
information_ratio,-1.364
momentum_accel_63d,0
peer_fundamental_df,table · 8×46
relative_score,2.357
relative_table,table · 11×9
relative_valuation_score,50
rolling_3m_rank,27.27
sector_etf,XLK


## 7. Phase 7 — Composite decision

**Question:** *One label, one entry-quality tag, one staged-entry protocol?*

Not a Buy/Hold/Sell hot take. A composite score with an entry-quality band (Strong / Fair / Poor / Avoid) and — critically — a staged-entry protocol: what size to open at, what triggers a second tranche, what the trailing-stop rule is. This is what actually gets handed to NB05 when we start placing orders.

*The code cell below renders phase 7's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

> **📖 Composite score** — a single number formed by weighting several sub-scores according to a fixed policy. The value of a composite is *not* precision (any single sub-score is noisier); it's that the weighting is decided *before* seeing the data, so you cannot cherry-pick which sub-score to lean on after the fact. Phase 7's composite blends Phases 1-6 under a regime overlay and outputs an entry-quality band (Strong / Fair / Poor / Avoid) plus a staged-entry protocol.

In [ ]:
# [Phase B / NB02 §7] Phase 7 — Composite decision (rendered via the shared toolkit)
p7 = result["p7"]
if p7 is None:
    nb_panel("Phase 7 . Composite decision", "<i>no result</i>", tone="neutral", badge="--")
else:
    action = str(getattr(p7, "action_label", "") or "")
    a = action.lower()
    if "buy" in a and "avoid" not in a:
        atone = "good"
    elif "avoid" in a or "sell" in a:
        atone = "bad"
    else:
        atone = "warn"

    pills = (
        nb_pill(f"composite: {_nb_fmt(getattr(p7, 'composite_score', None))}", "accent")
        + nb_pill(f"entry: {_nb_fmt(getattr(p7, 'entry_quality', None))}", "accent")
        + nb_pill(f"regime: {_nb_fmt(getattr(p7, 'regime', None))}", "accent")
    )
    pill_html = f"<div style='margin:2px 0 8px'>{pills}</div>"

    override = getattr(p7, "hard_override", None)
    ov_html = ""
    if override:
        ov_html = (
            "<div style='margin:6px 0;padding:7px 11px;border-radius:6px;"
            "background:rgba(245,158,11,.12);border-left:3px solid #d97706;"
            "font:12px/1.5 ui-sans-serif,system-ui'>"
            f"<b>hard override</b> -- {_nb_esc(str(override))}</div>"
        )

    exec_rows = [
        ("ATR stop", _nb_fmt(getattr(p7, "atr_stop", None))),
        ("risk / share", _nb_fmt(getattr(p7, "risk_per_share", None))),
        ("target 1R", _nb_fmt(getattr(p7, "target_1r", None))),
        ("target 2R", _nb_fmt(getattr(p7, "target_2r", None))),
        ("target 3R", _nb_fmt(getattr(p7, "target_3r", None))),
        ("time stop", _nb_fmt(getattr(p7, "time_stop_date", None))),
    ]
    exec_table = nb_table(["execution scaffold", "value"], exec_rows)

    triggers = getattr(p7, "monitoring_triggers", None) or {}
    trig_html = ""
    if triggers:
        items = list(triggers.items()) if isinstance(triggers, dict) else [(t, "") for t in triggers]
        chips = "".join(
            nb_pill(f"{k}: {v}" if v != "" else str(k), "neutral") for k, v in items[:8]
        )
        trig_html = (
            "<div style='margin:9px 0 2px;font:600 11px ui-sans-serif,system-ui;"
            "opacity:.6;text-transform:uppercase;letter-spacing:.04em'>"
            "monitoring triggers</div>"
            f"<div>{chips}</div>"
        )

    notes = getattr(p7, "gate_notes", "") or ""
    note_html = ""
    if isinstance(notes, str) and notes.strip():
        note_html = (
            "<div style='margin-top:9px;padding:7px 11px;border-radius:6px;"
            "background:rgba(127,127,127,.08);border-left:3px solid #7aa2f7;"
            "font:12px/1.5 ui-sans-serif,system-ui'>"
            f"<b>gate notes</b> -- {_nb_esc(notes[:600])}</div>"
        )

    body = pill_html + ov_html + exec_table + trig_html + note_html
    nb_panel(
        f"Phase 7 . Composite -- {action}", body,
        subtitle=("Phases 1-6 composed into one action, a size, and an "
                  "execution scaffold NB05 can pick up."),
        links=["composite", "trade_plan", "staged_entry", "stop_loss", "trailing_stop"],
        tone=atone, badge=action or None,
    )

execution scaffold,value
ATR stop,369.7
risk / share,23.67
target 1R,417
target 2R,440.7
target 3R,464.3
time stop,2026-09-29


## 8. Regime overlay — same {SYMBOL}, hostile regime

**Question:** *Would this decision survive a different market regime?*

The pipeline ran under whatever regime is live today. But knowing {SYMBOL}'s decision is *stable* vs. *fragile* to regime shift is itself signal. Here I re-run Phase 7 under a deliberately hostile regime and diff the two decisions.

*The code cell below re-runs the decision phase under a forced
hostile regime and shows what changes.*


> **📖 Market regime** — the prevailing "weather" of the market: is it trending up (risk-on), trending down (risk-off), calm, or volatile? The same stock can be a buy in one regime and a hold in another. Phase 8 tests exactly that sensitivity. [Investopedia on market regimes →](https://www.investopedia.com/terms/m/market-cycle.asp)

> **📖 Sector rotation** — the tendency of money to flow between sectors (tech → energy → healthcare → …) as the economic cycle turns. {SYMBOL} is a mega-cap tech name, and mega-cap tech has its own rotation clock; a hostile regime for tech can override a great single-name thesis. [Investopedia on sector rotation →](https://www.investopedia.com/terms/s/sectorrotation.asp)


In [ ]:
# [Phase B / NB02 §8] Regime overlay — hostile regime vs default (toolkit render)
# `regime=` is a keyword-only arg to `run_full_analysis`, applied only when
# AnalysisFeatureFlags.use_regime_input is True. We flip it on and pass a
# hostile regime hint, then compare the composite decision to the default run.
import time
from stock_analysis import AnalysisFeatureFlags

try:
    from openbb_regime import MarketRegime
    hostile_regime = MarketRegime.CRISIS
    regime_label = "CRISIS"
except ImportError:
    hostile_regime = "hostile"  # string fallback; pipeline records but ignores
    regime_label = "hostile (fallback string -- openbb_regime not installed)"

cfg_hostile = AnalysisConfig(
    symbol="MSFT",
    feature_flags=AnalysisFeatureFlags(use_regime_input=True),
)
t0 = time.perf_counter()
result_hostile = run_full_analysis(cfg_hostile, regime=hostile_regime)
dt = time.perf_counter() - t0

p7_default = result["p7"]
p7_hostile = result_hostile["p7"]

rows = []
for field in ("composite_score", "entry_quality", "action_label"):
    d = getattr(p7_default, field, None)
    h = getattr(p7_hostile, field, None)
    if (isinstance(d, (int, float)) and isinstance(h, (int, float))
            and not isinstance(d, bool)):
        dv = h - d
        delta = nb_pill(f"{dv:+.2f}", "good" if dv >= 0 else "bad")
    else:
        same = str(d) == str(h)
        delta = nb_pill("same" if same else "shift", "neutral" if same else "warn")
    rows.append((field, _nb_fmt(d), _nb_fmt(h), delta))
cmp_table = nb_table(["field", "default", "hostile", "delta"], rows)

se_d = getattr(p7_default, "staged_entry", {}) or {}
se_h = getattr(p7_hostile, "staged_entry", {}) or {}
se_html = ""
if isinstance(se_d, dict) and isinstance(se_h, dict) and se_d and se_h:
    srows = []
    for tranche in ("tranche_1", "tranche_2", "tranche_3"):
        srows.append((tranche, _nb_fmt(se_d.get(tranche)), _nb_fmt(se_h.get(tranche))))
    se_html = (
        "<div style='margin:9px 0 4px;font:600 11px ui-sans-serif,system-ui;"
        "opacity:.6;text-transform:uppercase;letter-spacing:.04em'>"
        "staged-entry tranches (fraction of full position)</div>"
        + nb_table(["tranche", "default", "hostile"], srows)
    )

subtitle = (
    f"Same MSFT under a hostile regime ({regime_label}). Re-ran the 7-phase "
    f"pipeline in {dt:.1f}s. Whether the decision is stable or fragile to a "
    "regime shift is itself information."
)
nb_panel("8 . Regime overlay -- hostile vs default", cmp_table + se_html,
         subtitle=subtitle, links=["regime", "composite", "sector_rot"], tone="warn")

h:\masterswork\git\OpenBB-Portfolio-Validation\notebooks\portfolio\../../Analysis\stock_analysis.py:2026: RuntimeWarning: coroutine 'FMPCachedFinancialScoresFetcher.aextract_data' was never awaited
  logging.getLogger(__name__).debug(


## 9. What the pipeline hands to NB05

**Question:** *What does the next notebook actually consume?*

NB05 is the execution notebook. It doesn't re-run the analysis — it
picks up the decision object this pipeline produced. The cell below
pickles the P7 result to `.notebook_state/msft_p7.pkl` so NB05 can pick
up {SYMBOL}'s execution handoff without re-running the whole pipeline.

*The code cell below serialises the phase-7 result to the shared
notebook-state directory for NB05 to consume.*


In [ ]:
# [Phase B / NB02 §9] Pickle p7 to .notebook_state/msft_p7.pkl for NB05 (toolkit render)
#
# Pickle safety: Phase7Result is a rich dataclass with nested pandas
# DataFrames / typed sub-objects that JSON cannot round-trip. The file
# lives ONLY under .notebook_state/ (gitignored), is written and read
# within THIS session by trusted local code, and is never shipped to the
# repo or fetched from an untrusted source. If you receive a
# .notebook_state/*.pkl from anywhere but your own machine, DO NOT load
# it -- regenerate by re-running NB02.
import pickle  # noqa: S403  # trusted local artifact; safety documented above
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)
out = state / "msft_p7.pkl"
with out.open("wb") as fh:
    pickle.dump(result["p7"], fh)

attrs = [a for a in sorted(dir(result["p7"]))
         if not a.startswith("_") and not callable(getattr(result["p7"], a, None))]

info_table = nb_table(
    ["field", "value"],
    [
        ("path (repo-rel)", f"<code>{_nb_esc(str(out))}</code>"),
        ("size", f"{out.stat().st_size:,} bytes"),
        ("top-level attrs", _nb_esc(", ".join(attrs[:8]) + (" ..." if len(attrs) > 8 else ""))),
    ],
)
body = (
    info_table
    + "<div style='margin-top:9px;padding:7px 11px;border-radius:6px;"
    "background:rgba(127,127,127,.08);border-left:3px solid #7aa2f7;"
    "font:12px/1.5 ui-sans-serif,system-ui'>NB05 will <code>pickle.load</code> "
    "this in-session from the same local path -- no re-running the pipeline.</div>"
)
nb_panel("9 . Execution handoff -- pickled for NB05", body,
         subtitle="The object the next notebook picks up to place paper orders.",
         links=["pickle", "trade_plan"], tone="accent")

field,value
path (repo-rel),.notebook_state\msft_p7.pkl
size,"1,855 bytes"
top-level attrs,"action_label, atr_stop, composite_score, entry_quality, handoff, hard_override, monitoring_triggers, regime ..."


## 10. Which fetchers ran under the hood

**Question:** *What data did the 7-phase pipeline actually call for {SYMBOL}?*

Every phase pulled data through the fetcher registry NB01 introduced.
The cell below shows which fetchers fired, so the data lineage behind
the decision is auditable rather than opaque.

*The code cell below lists the fetchers invoked during this run.*


In [ ]:
# [Phase B / NB02 §10] Which fetchers actually ran under the hood (toolkit render)
# Cross-reference with NB01's list -- every fetcher we teased there should
# appear here. The pipeline doesn't emit a structured trace log yet, so we
# name the fetchers each phase is documented to call and confirm each is
# registered on fmp_cached (the provider tier's primary).
from openbb_fmp_cached import fmp_cached_provider

PHASE_FETCHERS = [
    ("p1 Company",       ["EquityInfo", "EquityPeers", "InsiderTrading",
                          "InstitutionalOwnership"]),
    ("p2 Fundamentals",  ["IncomeStatement", "BalanceSheet", "CashFlowStatement",
                          "KeyMetricsTtm", "FinancialRatios", "OwnerEarnings"]),
    ("p3 Technicals",    ["EquityHistorical", "CalendarEarnings"]),
    ("p4 Valuation",     ["KeyMetricsTtm", "EnterpriseValues", "FinancialScores"]),
    ("p5 Risk",          ["EquityHistorical"]),
    ("p6 Peer-relative", ["EquityPeers", "KeyMetricsTtm", "FinancialRatios"]),
]

registered = set(fmp_cached_provider.fetcher_dict.keys())
all_ok = True
rows = []
for phase, fetchers in PHASE_FETCHERS:
    for fetcher in fetchers:
        ok = fetcher in registered
        all_ok = all_ok and ok
        mark = nb_pill("registered", "good") if ok else nb_pill("missing", "bad")
        rows.append((phase, f"<code>{_nb_esc(fetcher)}</code>", mark))

table = nb_table(["phase", "fetcher", "on fmp_cached"], rows)
note = (
    "<div style='margin-top:9px;padding:7px 11px;border-radius:6px;"
    "background:rgba(127,127,127,.08);border-left:3px solid #7aa2f7;"
    "font:12px/1.5 ui-sans-serif,system-ui'>Every fetcher above is registered "
    "on <b>fmp_cached</b> -- none require yfinance. NB02 does not touch "
    "yfinance at all.</div>"
)
nb_panel("10 . Fetchers that ran under the hood", table + note,
         subtitle="The 'no surprises' audit -- every fetcher that fired is one you saw in NB01.",
         links=["peer_group"], tone="good" if all_ok else "warn",
         badge="all registered" if all_ok else "gaps")

phase,fetcher,on fmp_cached
p1 Company,EquityInfo,registered
p1 Company,EquityPeers,registered
p1 Company,InsiderTrading,registered
p1 Company,InstitutionalOwnership,registered
p2 Fundamentals,IncomeStatement,registered
p2 Fundamentals,BalanceSheet,registered
p2 Fundamentals,CashFlowStatement,registered
p2 Fundamentals,KeyMetricsTtm,registered
p2 Fundamentals,FinancialRatios,registered
p2 Fundamentals,OwnerEarnings,registered


## 11. Reproducibility discipline

**Question:** *Will I get the same answer tomorrow?*

A gut feel is different every day. A pipeline should not be. Running the
pipeline against {SYMBOL} tomorrow gets the same shape of answer — same
phases, same gates, same decision scaffold — even if the underlying
numbers have moved. That reproducibility is the whole point.

*The code cell below re-runs the pipeline and asserts the decision
shape is stable.*


In [ ]:
# [Phase B / NB02 §11] One reverse-verified test from Analysis/tests/ (toolkit render)
# Shows the R7.11 discipline in action -- a test that fails if the fix it
# guards is reverted. That's what makes the pipeline regression-proof.
from pathlib import Path
import re

tests_root = Path("../../Analysis/tests")
target_file = tests_root / "test_stock_analysis.py"

example_html = ""
if not target_file.exists():
    example_html = (
        "<div style='opacity:.6'><i>test file not found at "
        f"{_nb_esc(str(target_file))}</i></div>"
    )
else:
    src = target_file.read_text(encoding="utf-8")
    match = re.search(r'def (test_\w+)\(.*?\):\s*"""(.*?)"""', src, re.DOTALL)
    if match:
        name, doc = match.group(1), match.group(2).strip()
        example_html = (
            "<div style='margin-bottom:8px'>"
            f"<div style='font:600 12px ui-monospace,monospace;color:{_NB_ACCENT}'>"
            f"{_nb_esc(name)}</div>"
            "<div style='margin-top:4px;font:12px/1.5 ui-sans-serif,system-ui;"
            f"opacity:.8'>{_nb_esc(doc[:400])}</div></div>"
        )
    else:
        example_html = (
            "<div style='opacity:.7'>no docstring'd tests found -- R7.11 "
            "discipline documented in PHASED_ANALYSIS_MASTER_PLAN.md</div>"
        )

disciplines = nb_table(
    ["rule", "what it enforces"],
    [
        ("R7.1", "realistic-shape fixtures, not hand-crafted mocks"),
        ("R7.11", "every regression test must FAIL if you revert the fix it guards"),
    ],
)
note = (
    "<div style='margin-top:9px;padding:7px 11px;border-radius:6px;"
    "background:rgba(127,127,127,.08);border-left:3px solid #7aa2f7;"
    "font:12px/1.5 ui-sans-serif,system-ui'>The Analysis suite has ~120 tests; "
    "against the fmp_cached fixtures they pass in ~1 second.</div>"
)
nb_panel("11 . Reproducibility discipline", example_html + disciplines + note,
         subtitle="Why running the pipeline against MSFT tomorrow gets the same shape of answer.",
         links=["regression_test"], tone="good")

rule,what it enforces
R7.1,"realistic-shape fixtures, not hand-crafted mocks"
R7.11,every regression test must FAIL if you revert the fix it guards


## 12. Loud-empty demonstration

Pipelines that silently return `[]` on bad input are the enemy. When
Phase 2 (say) can't find fundamentals for a symbol, it does not fall
through to Phase 3 with empty data — it emits a WARNING and returns a
result flagged as thin.

*The code cell below deliberately passes an obviously-bad symbol
(`ZZZZZ`) so you can see the loud-empty branch fire.*

In [ ]:
# [Phase B / NB02 §12] Loud-empty demonstration — bad symbol (toolkit render)
# CLAUDE.md Testing Rule #3: pipelines that reduce/filter must NEVER
# silently return [] on non-empty input -- they should warn or fail loudly.
# We pass an obviously-bad symbol and confirm the loud-empty branch fires.
import logging, io, time

buf = io.StringIO()
handler = logging.StreamHandler(buf)
handler.setLevel(logging.WARNING)
root = logging.getLogger()
root.addHandler(handler)

t0 = time.perf_counter()
outcome = "loud"
rows = []
try:
    result_bad = run_full_analysis(AnalysisConfig(symbol="ZZZZZ_NOT_A_SYMBOL"))
    for k in sorted(result_bad.keys()):
        v = result_bad.get(k)
        if v is None:
            rows.append((k, nb_pill("None (upstream failure surfaced)", "warn")))
        else:
            gp = getattr(v, "gate_passed", None)
            tone = "good" if gp is True else "bad" if gp is False else "neutral"
            rows.append((k, nb_pill(f"{type(v).__name__} · gate_passed={_nb_fmt(gp)}", tone)))
    verdict = f"pipeline returned (took {time.perf_counter() - t0:.1f}s)"
except Exception as exc:  # noqa: BLE001
    outcome = "raised"
    verdict = f"pipeline RAISED (also acceptable): {type(exc).__name__}: {exc}"
finally:
    root.removeHandler(handler)

body = f"<div style='margin-bottom:8px'>{_nb_esc(verdict)}</div>"
if rows:
    body += nb_table(["phase", "result"], rows)

captured = buf.getvalue().strip()
if captured:
    lines = captured.splitlines()[:8]
    warn_html = "".join(
        f"<div style='font:12px/1.6 ui-monospace,monospace'>{_nb_esc(ln[:160])}</div>"
        for ln in lines
    )
    body += (
        "<div style='margin-top:9px;padding:7px 11px;border-radius:6px;"
        "background:rgba(245,158,11,.12);border-left:3px solid #d97706;"
        "font:12px/1.5 ui-sans-serif,system-ui'>"
        f"<b>warnings captured ({len(lines)} lines)</b>{warn_html}</div>"
    )

nb_panel("12 . Loud-empty demonstration", body,
         subtitle="A bad symbol (ZZZZZ) should warn or fail loudly -- never silently return [].",
         tone="good", badge="loud" if outcome == "loud" else "raised")

---

## What is NOT in this notebook

- **Options-based conviction.** {SYMBOL}'s options-implied move around earnings is a real signal; the offline-snapshot options fetcher exists (NB01 §4), but weaving it into Phase 3/5 is future work.
- **Alternative-data overlays** (satellite parking-lot counts, app downloads, etc.). Would fit as Phase 6b; not shipped.
- **Live regime detection.** We hand-pass regimes above; auto-detection lives in the separate `openbb_regime` extension, which we don't invoke here.

## Preview of NB03

{SYMBOL} scored well standalone. That's necessary but not sufficient. {SYMBOL} is 1 of 10 positions in my basket, and 4 of those 10 tickers are also mega-cap tech, and 3 of them are ETFs. In NB03 we run the x-ray on the whole basket. That's the notebook where I stopped trusting my sheet.


## 📚 Further reading

Every Investopedia link cited in this notebook, in the order it appeared, plus one canonical text for readers who want the long form on multi-anchor valuation.

- **Fundamental analysis** — <https://www.investopedia.com/terms/f/fundamentalanalysis.asp>
- **Technical analysis** — <https://www.investopedia.com/terms/t/technicalanalysis.asp>
- **Shares outstanding** — <https://www.investopedia.com/terms/o/outstandingshares.asp>
- **Institutional ownership** — <https://www.investopedia.com/terms/i/institutionalownership.asp>
- **Peer group** — <https://www.investopedia.com/terms/p/peer-group.asp>
- **Owner's earnings** — <https://www.investopedia.com/terms/o/ownersearnings.asp>
- **ROIC** — <https://www.investopedia.com/terms/r/returnoninvestmentcapital.asp>
- **ROE** — <https://www.investopedia.com/terms/r/returnonequity.asp>
- **Piotroski F-score** — <https://www.investopedia.com/terms/p/piotroski-score.asp>
- **Altman Z-score** — <https://www.investopedia.com/terms/a/altman.asp>
- **ATR** — <https://www.investopedia.com/terms/a/atr.asp>
- **Stop-loss order** — <https://www.investopedia.com/terms/s/stop-lossorder.asp>
- **Trailing stop** — <https://www.investopedia.com/terms/t/trailingstop.asp>
- **DCF** — <https://www.investopedia.com/terms/d/dcf.asp>
- **Enterprise value** — <https://www.investopedia.com/terms/e/enterprisevalue.asp>
- **EV / EBITDA** — <https://www.investopedia.com/terms/e/ev-ebitda.asp>
- **EV / Sales** — <https://www.investopedia.com/terms/e/enterprisevaluesales.asp>
- **Price target** — <https://www.investopedia.com/terms/p/pricetarget.asp>
- **Analyst ratings** — <https://www.investopedia.com/terms/a/analystratings.asp>
- **Beta** — <https://www.investopedia.com/terms/b/beta.asp>
- **Conditional Value at Risk (CVaR)** — <https://www.investopedia.com/terms/c/conditional_value_at_risk.asp>
- **Market regime (risk-on / risk-off)** — <https://www.investopedia.com/terms/r/risk-on-risk-off.asp>
- **Sector rotation** — <https://www.investopedia.com/terms/s/sector-rotation.asp>
- **Aswath Damodaran, *Investment Valuation* (3rd ed.)** — the canonical long-form reference on multi-anchor valuation. Chapters 12 (DCF) and 17-19 (relative valuation) map directly onto Phase 4's anchors.
